<a href="https://colab.research.google.com/github/AsiminaXi/BRCA_SiameseMLP/blob/main/BrcaSiameseMlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries
!pip install fair-esm biopython umap-learn --quiet

In [ ]:
# Download data from Google Drive
import gdown, zipfile, os, shutil

%cd /content

file_ids = ["17wSB_xFZRomAuhJI3kYr2TwVYBm4Y5Wz"]
output_dir = "downloads"
os.makedirs(output_dir, exist_ok=True)

for file_id in file_ids:
    url = f"https://drive.google.com/uc?id={file_id}"
    output_path = os.path.join(output_dir, f"{file_id}.zip")
    print(f"Downloading {file_id}...")
    gdown.download(url, output_path, quiet=False)
    if zipfile.is_zipfile(output_path):
        with zipfile.ZipFile(output_path, "r") as z:
            z.extractall(os.path.join(output_dir, file_id))

# Download labels file
file_id_labels = "1aEDUaVubhT24WDCxMUnhit5BOaqXjb6l"
gdown.download(
    f"https://drive.google.com/uc?id={file_id_labels}",
    os.path.join(output_dir, "training_labels.txt"),
    quiet=False,
)


In [ ]:
# Organize folders
import os, shutil

brca_dir = "/content/brca1_2"
if os.path.exists(brca_dir):
    shutil.rmtree(brca_dir)

input_dir = "/content/input"
os.makedirs(input_dir, exist_ok=True)
source_dir = "/content/downloads/17wSB_xFZRomAuhJI3kYr2TwVYBm4Y5Wz/input_to_CNN"

# Copy only aligned FASTA files
for fname in os.listdir(source_dir):
    if fname.endswith(".fasta") and "aligned" in fname:
        shutil.copy2(os.path.join(source_dir, fname), os.path.join(input_dir, fname))

os.rename(input_dir, brca_dir)
fasta_count = sum(1 for f in os.listdir(brca_dir) if f.endswith(".fasta"))
print(f"FASTA files: {fasta_count}")

In [ ]:
#  Merge groups — SYNCED SEQ-ANCHOR METHOD (OPTIMIZED)
#
#  Logic:
#  - seq_X as anchor.
#  - Filter homologs < 30% of seq_X length.
#  - Deduplication via accession ID.
#  - **SYNCED FILTERING**. Ensures that the exact same set of
#    homologs (Accession IDs) is kept in BOTH the WT and MUT files!

import os, re
from collections import OrderedDict

def read_fasta(filepath):
    seqs = OrderedDict()
    header, buf = None, []
    with open(filepath) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith(">"):
                if header:
                    seqs[header] = "".join(buf)
                header, buf = line[1:], []
            else:
                buf.append(line)
    if header:
        seqs[header] = "".join(buf)
    return seqs

def find_seq_key(seqs):
    for k in seqs:
        if re.match(r"^seq_\d+$", k.strip()):
            return k
    return None

def trim_to_anchor(seqs, anchor_key):
    anchor = seqs[anchor_key]
    keep = [i for i, aa in enumerate(anchor) if aa != "-"]
    return {h: "".join(s[i] for i in keep if i < len(s)) for h, s in seqs.items()}

def get_valid_homologs(group_files):
    """
    Reads group files and returns:
    1. The seq_key (e.g., 'seq_131')
    2. The clean sequence of seq_X
    3. A dictionary of valid homologs: {accession_id: (full_header, clean_seq)}
    """
    homologs = OrderedDict()
    seq_key = seq_seq = None

    for fp in group_files:
        if not os.path.exists(fp):
            continue
        seqs = read_fasta(fp)
        if not seqs:
            continue
        sk = find_seq_key(seqs)
        if sk is None:
            continue

        if seq_key is None:
            seq_key = sk
            seq_seq  = seqs[sk].replace("-", "")

        trimmed = trim_to_anchor(seqs, sk)

        for header, seq in trimmed.items():
            if re.match(r"^seq_\d+$", header.strip()):
                continue
            accession = header.split()[0]
            if accession in homologs:
                continue

            clean = seq.replace("-", "")
            # 30% length filter
            if len(clean) < 0.30 * len(seq_seq):
                continue

            homologs[accession] = (header, clean)

    return seq_key, seq_seq, homologs

def merge_all_genes_synced(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    # Find all pairs (seq_1, seq_2, etc.) per gene (brca1, brca2)
    # We will group based on {gene}_{seq_number} (e.g., brca1_seq_131)
    pattern = re.compile(r"(brca[12])[ab]_aligned_(seq_\d+)_group[123]\.fasta$", re.I)

    pairs = {} # { 'brca1_seq_131': {'a': [...files], 'b': [...files]} }

    for fname in os.listdir(input_dir):
        m = pattern.match(fname)
        if m:
            gene = m.group(1).lower() # brca1 or brca2
            seq_id = m.group(2).lower() # seq_131
            pair_key = f"{gene}_{seq_id}"
            variant_type = 'a' if 'a_aligned' in fname.lower() else 'b'

            if pair_key not in pairs:
                pairs[pair_key] = {'a': [], 'b': []}
            pairs[pair_key][variant_type].append(os.path.join(input_dir, fname))

    ok = fail = 0

    for pair_key, files in pairs.items():
        gene, seq_id = pair_key.split('_', 1)

        if not files['a'] or not files['b']:
            continue # Missing 'a' or 'b' entirely, ignore

        # 1. Find valid homologs for WT (a) and MUT (b) independently
        a_key, a_seq, a_homologs = get_valid_homologs(files['a'])
        b_key, b_seq, b_homologs = get_valid_homologs(files['b'])

        if not a_key or not b_key:
            fail += 1
            print(f"  FAIL: {pair_key} (Missing seq_X)")
            continue

        # 2. **SYNCED FILTERING**: Find the intersection of Accession IDs!
        common_accessions = set(a_homologs.keys()).intersection(set(b_homologs.keys()))

        # 3. Write the WT file, keeping ONLY common_accessions
        out_a = os.path.join(output_dir, f"{gene}a_aligned_{seq_id}.fasta")
        with open(out_a, "w") as f:
            f.write(f">{a_key}\n{a_seq}\n")
            for acc in common_accessions:
                hdr, seq = a_homologs[acc]
                f.write(f">{hdr}\n{seq}\n")

        # 4. Write the MUT file, keeping ONLY common_accessions
        out_b = os.path.join(output_dir, f"{gene}b_aligned_{seq_id}.fasta")
        with open(out_b, "w") as f:
            f.write(f">{b_key}\n{b_seq}\n")
            for acc in common_accessions:
                hdr, seq = b_homologs[acc]
                f.write(f">{hdr}\n{seq}\n")

        ok += 2 # Successfully wrote 2 files (WT and MUT)

    print(f"\nSynced Merge complete: {ok} files written · {fail} FAIL")

# Run the new function
merge_all_genes_synced("/content/brca1_2", "/content/merged_fasta")

# The file counting code remains the same below...

In [ ]:
import os
import pandas as pd

def count_fasta_files(directory, label=None):
    """
    Count FASTA files per gene (brca1a, brca1b, brca2a, brca2b).
    Returns a dict and prints a summary report.
    """
    counts = {
        "brca1a": 0,
        "brca1b": 0,
        "brca2a": 0,
        "brca2b": 0,
    }

    if not os.path.exists(directory):
        print(f" The folder '{directory}' does not exist.")
        return counts

    fasta_files = [f for f in os.listdir(directory) if f.lower().endswith((".fasta", ".fa"))]

    for filename in fasta_files:
        lower_name = filename.lower()
        for gene in counts:
            if gene in lower_name:
                counts[gene] += 1
                break

    total = sum(counts.values())
    label_text = f" ({label})" if label else ""

    print(f"\nReport for folder: {directory}{label_text}")
    for gene, count in counts.items():
        print(f"  • {gene}: {count}")
    print(f"Total files: {total}")

    return counts


def compare_fasta_counts(folder_dict):
    """
    Compare FASTA file counts across multiple directories.
    Returns a summary DataFrame.
    """
    all_counts = {}

    for label, path in folder_dict.items():
        all_counts[label] = count_fasta_files(path, label=label)

    df = pd.DataFrame(all_counts)
    df.loc["Total"] = df.sum()

    print("\n === FASTA File Comparison Table ===")
    display(df)
    return df


folders = {
    "Original (brca1_2)": "/content/brca1_2",
    "Merged": "/content/merged_fasta"
}

counts_df = compare_fasta_counts(folders)


In [ ]:
#  Verify WT ↔ MUT pairs
#
#  Checks that:
#  - Each WT file has a corresponding MUT file
#  - Both files have the same number of sequences
#  - The query sequences (seq_X) are the same length
#  - Exactly one amino acid position differs (the missense mutation)
!pip install biopython --quiet
import os
from Bio import SeqIO

def verify_pairs(directory):
    """
    Verify WT/MUT FASTA pairs for structural integrity.
    Returns a list of problematic WT filenames.
    """
    total = perfect = problems = 0

    for fname in sorted(os.listdir(directory)):
        if not (fname.startswith(("brca1a_", "brca2a_")) and fname.endswith(".fasta")):
            continue
        wt_path  = os.path.join(directory, fname)
        mut_path = os.path.join(directory, fname.replace("a_", "b_", 1))
        if not os.path.exists(mut_path):
            print(f"  MISSING: {fname.replace('a_','b_',1)}")
            continue
        total += 1

        wt_seqs  = list(SeqIO.parse(wt_path,  "fasta"))
        mut_seqs = list(SeqIO.parse(mut_path, "fasta"))

        # Index 0 is always seq_X after merge
        wt_main  = str(wt_seqs[0].seq)  if wt_seqs  else ""
        mut_main = str(mut_seqs[0].seq) if mut_seqs else ""

        mut_positions = [i for i,(a,b) in enumerate(zip(wt_main, mut_main)) if a != b]
        n_mut = len(mut_positions)

        ok = (len(wt_seqs) == len(mut_seqs) and len(wt_main) == len(mut_main) and n_mut == 1)
        if ok:
            perfect += 1
        else:
            problems += 1
            print(f"  FAIL: {fname}: {len(wt_seqs)} vs {len(mut_seqs)} seqs | "
                  f"len {len(wt_main)} vs {len(mut_main)} | mutations={n_mut}")


    rate = (perfect/total*100) if total else 0
    print(f"\nTotal: {total} | OK: {perfect} | Problems: {problems} | {rate:.1f}%")

verify_pairs("/content/merged_fasta")

In [ ]:
#  Build dataset CSV
#
#  Creates merged_msa_data.csv with columns:
#  WT_MSA, MUT_MSA, Label, Gene
#  Problematic pairs identified in Cell 5 are excluded.
import os, re, pandas as pd
import gdown

def extract_seq_number(fname):
    """Extract the numeric index from a seq_X filename for sorting."""
    m = re.search(r"seq_(\d+)", fname)
    return int(m.group(1)) if m else -1

def build_dataset_csv(merged_dir, labels_path, out_csv):
    """
    Build the paired dataset CSV from merged FASTA files and label file.
    Filters out problematic pairs found during verification.
    """
    # Re-download labels if missing
    if not os.path.exists(labels_path):
        print(f"  WARNING: Labels file '{labels_path}' not found. Attempting re-download...")
        # Use the file_id from CELL 2 for labels
        file_id_labels = "1aEDUaVubhT24WDCxMUnhit5BOaqXjb6l"
        gdown.download(
            f"https://drive.google.com/uc?id={file_id_labels}",
            labels_path,
            quiet=False,
        )
        if not os.path.exists(labels_path):
            raise FileNotFoundError(f"Cannot find or download labels file: {labels_path}")
        else:
            print(f"Labels file downloaded successfully.")

    with open(labels_path) as f:
        labels = [l.strip() for l in f if l.strip()]

    wt_files  = sorted(
        [f for f in os.listdir(merged_dir) if re.match(r"brca[12]a_", f)],
        key=lambda x: (x[:6], extract_seq_number(x))
    )
    mut_files = sorted(
        [f for f in os.listdir(merged_dir) if re.match(r"brca[12]b_", f)],
        key=lambda x: (x[:6], extract_seq_number(x))
    )

    # The original assert checks the initial state of files/labels lists.
    # Filtering will happen after DataFrame creation.
    assert len(wt_files) == len(mut_files) == len(labels), \
        f"Mismatch: WT={len(wt_files)}, MUT={len(mut_files)}, Labels={len(labels)}"

    df = pd.DataFrame({
        "WT_MSA" : wt_files,
        "MUT_MSA": mut_files,
        "Label"  : labels,
        "Gene"   : ["BRCA1" if "brca1" in f.lower() else "BRCA2" for f in wt_files],
    })


    df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}  ({len(df)} pairs)")
    print(df["Label"].value_counts())
    return df

df_meta = build_dataset_csv(
    "/content/merged_fasta",
    "/content/downloads/training_labels.txt",
    "/content/merged_msa_data.csv",
)

In [ ]:
#  ESM-MSA-1b Embedding Extraction
#
#  For each WT/MUT FASTA pair:
#  - Loads the merged MSA
#  - Pads/trims homologs to query (seq_X) length
#  - Extracts mean per-residue embedding from layer 12 → (768,)
#  - Computes mean log-likelihood score (LLR) for seq_X
#  - Caches results as .npy files to avoid re-computation
import torch, numpy as np, pandas as pd, esm, os
from Bio import SeqIO
from pathlib import Path
from tqdm.auto import tqdm

# Load pretrained ESM-MSA-1b model
print("Loading ESM-MSA-1b...")
model, alphabet = esm.pretrained.esm_msa1b_t12_100M_UR50S()
batch_converter  = alphabet.get_batch_converter()
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)
print(f"Running on: {device}\n")

MAX_SEQS = 512   # Hard limit for ESM-MSA-1b
MAX_LEN  = 1022  # Leaves room for [CLS] and [EOS] tokens


def prepare_aligned_msa(fasta_path):
    """
    Read a FASTA file and return an aligned MSA as list of (id, seq) tuples.

    Rules:
    - First sequence (seq_X) is the query — gaps removed.
    - All homologs are padded or trimmed to match query length.
    - Shorter homologs → right-padded with '-'.
    - Longer homologs → truncated to query length.
    """
    records = list(SeqIO.parse(fasta_path, "fasta"))
    if not records:
        return None

    # Query sequence: remove any gaps, apply length limit
    query_seq = str(records[0].seq).replace("-", "")
    query_len = len(query_seq)

    # Truncation query
    if query_len > MAX_LEN:
        query_seq = query_seq[:MAX_LEN]
        query_len = MAX_LEN

    msa = [(records[0].id, query_seq)]

    # Pad or trim homologs to query length (preserve alignment gaps)
    for rec in records[1:]:
        seq = str(rec.seq)  # Preserve alignment gaps — do not strip gaps from homologs
        if len(seq) < query_len:
            seq = seq + "-" * (query_len - len(seq))
        elif len(seq) > query_len:
            seq = seq[:query_len]                       # truncate
        msa.append((rec.id, seq))

    # Verify uniform length after padding
    lengths = set(len(s) for _, s in msa)
    assert len(lengths) == 1, f"Lengths mismatch after padding: {lengths}"

    # Cap number of sequences
    if len(msa) > MAX_SEQS:
        msa = msa[:MAX_SEQS]

    return msa


def get_embedding_and_score(fasta_path, model, batch_converter, save_dir):
    """
    Extract ESM-MSA-1b embedding and log-likelihood score for a FASTA file.

    Saves:
      - *_emb.npy   : mean per-residue embedding of seq_X  shape (768,)
      - *_score.npy : mean log-likelihood of seq_X          shape (1,)

    Returns paths to saved files, or (None, None) on failure.
    """
    fasta_path = Path(fasta_path)
    emb_path   = Path(save_dir) / f"{fasta_path.stem}_emb.npy"
    score_path = Path(save_dir) / f"{fasta_path.stem}_score.npy"

    # Cache
    if emb_path.exists() and score_path.exists():
        return str(emb_path)

    if not fasta_path.exists():
        print(f"  MISSING: {fasta_path.name}")
        return None, None

    msa = prepare_aligned_msa(fasta_path)
    if msa is None or len(msa) == 0:
        print(f"  EMPTY: {fasta_path.name}")
        return None, None

    try:
        _, _, tokens = batch_converter([msa])
        tokens = tokens.to(device)

        with torch.no_grad():
            out = model(tokens, repr_layers=[12], return_contacts=False)

        # Extract mean per-residue embedding for seq_X (index 0)
        # Shape: [1, num_seqs, seq_len+2, 768] → remove [CLS]/[EOS]
        repr_layer = out["representations"][12]
        seq_repr   = repr_layer[0, 0, 1:-1, :]         # [L, 768]
        embedding  = seq_repr.mean(dim=0).cpu().numpy()  # (768,)
        np.save(emb_path,   embedding)
        return str(emb_path)


    except Exception as e:
        print(f"\n  ERROR {fasta_path.name}: {e}")
        return None


# Sanity check on first file before running full extraction
print("Running sanity check on first file...")
df_meta   = pd.read_csv("/content/merged_msa_data.csv")
msa_dir   = Path("/content/merged_fasta")
save_dir  = Path("/content/esm_embeddings")
save_dir.mkdir(exist_ok=True)

test_file = msa_dir / df_meta["WT_MSA"].iloc[0]
test_msa  = prepare_aligned_msa(test_file)
if test_msa:
    lengths = set(len(s) for _, s in test_msa)
    print(f"  File: {test_file.name}")
    print(f"  Sequences: {len(test_msa)}")
    print(f"  Query length: {len(test_msa[0][1])}")
    print(f"  Unique lengths: {lengths}  ← must be exactly 1 value")
    print("  Sanity check OK!\n")

# Run embedding extraction for all WT/MUT pairs
wt_embs, mut_embs = [], []

for _, row in tqdm(df_meta.iterrows(), total=len(df_meta), desc="Embeddings"):
    we = get_embedding_and_score(msa_dir / row["WT_MSA"],  model, batch_converter, save_dir)
    me = get_embedding_and_score(msa_dir / row["MUT_MSA"], model, batch_converter, save_dir)
    wt_embs.append(we)
    mut_embs.append(me)


df_meta["WT_Embedding"]  = wt_embs
df_meta["MUT_Embedding"] = mut_embs

# Remove rows where embedding extraction failed
n_before = len(df_meta)
df_meta  = df_meta.dropna(subset=["WT_Embedding", "MUT_Embedding"])
n_after  = len(df_meta)
print(f"\nSaved: {n_after} pairs ({n_before - n_after} failures removed)")

df_meta.to_csv("/content/siamese_dataset.csv", index=False)
print("siamese_dataset.csv saved!")

# Preview first rows
print("\nSample:")
print(df_meta[["WT_MSA", "MUT_MSA", "Label"]].head())



In [ ]:
#  Site-Specific ΔLLR Computation
#
#    ΔLLR = log P(mut_aa | pos, WT_context) - log P(wt_aa | pos, WT_context)
#
#  - Context: ALWAYS the WT MSA (frozen)
#  - Position: Only the single residue that differs
#  - Cache: Saves *_site_score.npy files

score_dir = Path("/content/esm_site_scores")
score_dir.mkdir(exist_ok=True)


def compute_site_delta_llr(wt_fasta, mut_fasta, model, batch_converter,
                            alphabet, device, score_dir):
    wt_fasta  = Path(wt_fasta)
    mut_fasta = Path(mut_fasta)

    cache_path = score_dir / f"{wt_fasta.stem}_site_score.npy"
    if cache_path.exists():
        return float(np.load(cache_path)[0])

    if not wt_fasta.exists() or not mut_fasta.exists():
        return None

    wt_msa  = prepare_aligned_msa(wt_fasta)
    mut_msa = prepare_aligned_msa(mut_fasta)
    if wt_msa is None or mut_msa is None:
        return None

    wt_seq  = wt_msa[0][1]
    mut_seq = mut_msa[0][1]

    diffs = [i for i, (a, b) in enumerate(zip(wt_seq, mut_seq)) if a != b]
    if len(diffs) != 1:
        print(f"  SKIP {wt_fasta.name}: {len(diffs)} mutation sites (expected 1)")
        return None

    pos    = diffs[0]
    wt_aa  = wt_seq[pos]
    mut_aa = mut_seq[pos]

    try:
        _, _, tokens = batch_converter([wt_msa])
        tokens = tokens.to(device)
        with torch.no_grad():
            out = model(tokens, repr_layers=[], return_contacts=False)

        logits_at_pos = out["logits"][0, 0, pos + 1, :]
        log_probs     = torch.log_softmax(logits_at_pos, dim=-1)

        wt_idx  = alphabet.get_idx(wt_aa)
        mut_idx = alphabet.get_idx(mut_aa)

        if wt_idx == alphabet.unk_idx or mut_idx == alphabet.unk_idx:
            print(f"  SKIP {wt_fasta.name}: unknown AA ({wt_aa}→{mut_aa})")
            return None


        delta = (log_probs[wt_idx] - log_probs[mut_idx]).item()
        np.save(cache_path, np.array([delta]))
        return delta

    except Exception as e:
        print(f"  ERROR {wt_fasta.name}: {e}")
        return None


df_meta = pd.read_csv("/content/siamese_dataset.csv")
msa_dir = Path("/content/merged_fasta")

site_scores = []
for _, row in tqdm(df_meta.iterrows(), total=len(df_meta), desc="Site ΔLLR"):
    score = compute_site_delta_llr(
        msa_dir / row["WT_MSA"],
        msa_dir / row["MUT_MSA"],
        model, batch_converter, alphabet, device, score_dir
    )
    site_scores.append(score)

df_meta["Site_ΔLLR"] = site_scores

n_before = len(df_meta)
df_meta  = df_meta.dropna(subset=["Site_ΔLLR"])
print(f"Success: {len(df_meta)}/{n_before} pairs")

delta_site = df_meta["Site_ΔLLR"].values
print(f"\nSite-Specific ΔLLR statistics:")
print(f"  Mean : {delta_site.mean():.4f}")
print(f"  Std  : {delta_site.std():.4f}")
print(f"  Pathogenic mean: {delta_site[df_meta['Label']=='Pathogenic'].mean():.4f}  ← negative is expected")
print(f"  Benign mean    : {delta_site[df_meta['Label']=='Benign'].mean():.4f}   ← expected close to 0")

df_meta.to_csv("/content/siamese_dataset.csv", index=False)
print("\nsiamese_dataset.csv updated με Site_ΔLLR column!")

In [ ]:
#  Siamese MLP Training — 5-Fold Cross-Validation
#
#  Architecture:
#  - Shared encoder: Linear → BN → GELU → Dropout → Linear → BN → GELU
#  - Interaction: |wt_enc - mut_enc| concatenated with wt_enc * mut_enc + ΔLLR
#  - Classifier: Linear → GELU → Dropout → Linear → sigmoid
#
#  Training details:
#  - Loss: BCE (with class weights + label smoothing) + Contrastive loss
#  - Optimizer: AdamW with CosineAnnealingLR scheduler
#  - Early stopping: patience=15 on validation loss
#  - Threshold: F1-optimized on validation set per fold
#  - Evaluation: Accuracy, F1, AUC-ROC aggregated across all folds
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, pandas as pd, copy
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.metrics import precision_recall_curve
import pandas as pd
import re
from google.colab import drive
drive.mount('/content/drive')

def extract_seq_id_from_filename(fname):
    m = re.search(r"seq_(\d+)", str(fname))
    return int(m.group(1)) if m else None

meta = pd.read_csv("/content/siamese_dataset.csv").dropna(
    subset=["WT_Embedding", "MUT_Embedding", "Site_ΔLLR"]
).copy()

def extract_full_seq_id_from_filename(fname):
    fname = str(fname).lower()

    gene_match = re.search(r"(brca[12])", fname)
    seq_match = re.search(r"seq_(\d+)", fname)

    if gene_match and seq_match:
        return f"{gene_match.group(1)}_seq_{seq_match.group(1)}"

    return None

meta["seq_id"] = meta["WT_MSA"].apply(extract_full_seq_id_from_filename)
print("Samples after filtering:", len(meta))

# Load embeddings and LLR scores
#meta = pd.read_csv("/content/siamese_dataset.csv").dropna(subset=["WT_Embedding","MUT_Embedding"])

WT_raw  = np.vstack([np.load(p) for p in meta["WT_Embedding"]]).astype(np.float32)
MUT_raw = np.vstack([np.load(p) for p in meta["MUT_Embedding"]]).astype(np.float32)

# ΔLLR: log P(mut|context) - log P(wt|context)
delta_llr_raw = meta["Site_ΔLLR"].values.astype(np.float32).reshape(-1, 1)

le = LabelEncoder()
y_raw  = le.fit_transform(meta["Label"].values).astype(np.float32)
seq_ids_raw = meta["seq_id"].values

# =========================================================================
# ADDITION: Keep 10% Hold-out Test Set exactly like Code 2
# (Using random_state=2 to ensure the EXACT SAME samples are used)
# =========================================================================

WT, WT_test_final, MUT, MUT_test_final, delta_llr, delta_llr_test_final, y, y_test_final, seq_train, seq_test = train_test_split(
    WT_raw, MUT_raw, delta_llr_raw, y_raw, seq_ids_raw, test_size=0.1, stratify=y_raw, random_state=2
)

shared_path = "/content/drive/MyDrive/shared_test_seq_ids.csv"

pd.DataFrame({
    "seq_id": sorted(seq_test)
}).to_csv(shared_path, index=False)

print(f"Saved shared test IDs to: {shared_path}")

print("Saved mlp_test_seq_ids.csv")
print(f"Classes: {le.classes_}")
print(f"Dataset for CV (90%): {len(y)} samples")
print(f"Dataset for Final Test (10%): {len(y_test_final)} samples")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class SiameseESM(nn.Module):
    """
    Siamese MLP operating on ESM-MSA-1b embeddings.

    Encoder processes WT and MUT independently.
    Classifier receives: [|wt_enc - mut_enc|, wt_enc * mut_enc, ΔLLR]
    """
    def __init__(self, in_dim=768, hidden=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.GELU(),
        )

        # Input: diff (128) + product (128) + ΔLLR scalar (1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden // 2 * 2 + 1, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, wt, mut, delta_llr):
        wt_enc  = self.encoder(wt)
        mut_enc = self.encoder(mut)
        diff    = torch.abs(wt_enc - mut_enc)
        prod    = wt_enc * mut_enc
        feat    = torch.cat([diff, prod, delta_llr], dim=1)
        logit   = self.classifier(feat)
        return logit, wt_enc, mut_enc


def contrastive_loss(z1, z2, y, margin=1.0):
    """
    Contrastive loss encouraging similar pairs to be close
    and dissimilar pairs to be at least margin apart.
    """
    d = torch.norm(z1 - z2, dim=1)
    loss = y.squeeze() * d.pow(2) + \
           (1 - y.squeeze()) * torch.clamp(margin - d, min=0.0).pow(2)
    return loss.mean()


# 5-Fold Stratified Cross-Validation (on the 90% CV dataset)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results, fold_epochs = [], []

# Collect predictions from all folds for aggregated evaluation
all_y_te = []
all_y_pred = []
all_te_prob = []
all_wt_te_embeddings = []
all_mut_te_embeddings = []
best_global_auc = -np.inf
best_global_state = None
best_global_fold  = -1
best_global_thr = 0.5 # Default


for fold, (tv_idx, test_idx) in enumerate(skf.split(WT, y)):
    print(f"\n{'='*48}\n  FOLD {fold+1}/5\n{'='*48}")

    WT_tv,  WT_te_raw   = WT[tv_idx],        WT[test_idx]
    MUT_tv, MUT_te_raw  = MUT[tv_idx],       MUT[test_idx]
    DL_tv,  DL_te   = delta_llr[tv_idx], delta_llr[test_idx]
    y_tv,   y_te    = y[tv_idx],         y[test_idx]

    # Split train/validation (90/10 of the non-test portion)
    WT_tr, WT_va, MUT_tr, MUT_va, DL_tr, DL_va, y_tr, y_va = train_test_split(
        WT_tv, MUT_tv, DL_tv, y_tv,
        test_size=0.1111, stratify=y_tv, random_state=42
    )

    # Fit scaler on training WT only — transform all splits with same scaler
    sc_emb = StandardScaler()

    WT_tr  = sc_emb.fit_transform(WT_tr)
    WT_va  = sc_emb.transform(WT_va)
    WT_te  = sc_emb.transform(WT_te_raw)

    MUT_tr = sc_emb.transform(MUT_tr)
    MUT_va = sc_emb.transform(MUT_va)
    MUT_te = sc_emb.transform(MUT_te_raw)

    sc_llr = StandardScaler()
    DL_tr  = sc_llr.fit_transform(DL_tr);  DL_va  = sc_llr.transform(DL_va);  DL_te  = sc_llr.transform(DL_te)

    def to_tensor(*arrays):
        return [torch.tensor(a).to(device) for a in arrays]

    WT_tr,  MUT_tr,  DL_tr,  y_tr_t  = to_tensor(WT_tr,  MUT_tr,  DL_tr,  y_tr.reshape(-1,1))
    WT_va,  MUT_va,  DL_va,  y_va_t  = to_tensor(WT_va,  MUT_va,  DL_va,  y_va.reshape(-1,1))
    WT_te_tensor,  MUT_te_tensor,  DL_te_tensor,  y_te_tensor  = to_tensor(WT_te,  MUT_te,  DL_te,  y_te.reshape(-1,1))

    # Weighted BCE to handle class imbalance
    n_pos = (y_tr == 1).sum(); n_neg = (y_tr == 0).sum()
    pos_w = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_w)

    net       = SiameseESM().to(device)
    optimizer = optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150)

    best_val, best_state, best_ep = np.inf, None, 0
    patience_cnt = 0
    PATIENCE = 15
    MIN_DELTA = 1e-4
    ALPHA    = 0.25

    for epoch in range(150):
        net.train()
        optimizer.zero_grad()
        logits, z1, z2 = net(WT_tr, MUT_tr, DL_tr)
        EPS = 0.05
        y_tr_smooth = y_tr_t * (1 - EPS) + EPS / 2
        loss = bce(logits, y_tr_smooth) + ALPHA * contrastive_loss(z1, z2, y_tr_t)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        net.eval()
        with torch.no_grad():
            vl, vz1, vz2 = net(WT_va, MUT_va, DL_va)
            y_va_smooth = y_va_t * (1 - EPS) + EPS / 2
            val_loss = bce(vl, y_va_smooth) + ALPHA * contrastive_loss(vz1, vz2, y_va_t)



        if val_loss.item() < best_val - MIN_DELTA:
            best_val = val_loss.item()
            best_state = copy.deepcopy(net.state_dict())
            best_ep = epoch + 1
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"  Early stopping @ epoch {epoch+1}  (best: {best_ep})")
            break

    fold_epochs.append(best_ep)
    net.load_state_dict(best_state)

    net.eval()
    with torch.no_grad():
        va_prob = torch.sigmoid(net(WT_va, MUT_va, DL_va)[0]).cpu().numpy().ravel()
    pr, rc, thr = precision_recall_curve(y_va, va_prob)
    f1s         = 2*pr*rc / (pr + rc + 1e-8)
    best_thr    = thr[np.argmax(f1s)]
    print(f"  Optimal threshold: {best_thr:.3f}")

    with torch.no_grad():
        te_prob = torch.sigmoid(net(WT_te_tensor, MUT_te_tensor, DL_te_tensor)[0]).cpu().numpy().ravel()
    y_pred = (te_prob > best_thr).astype(int)

    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred, zero_division=0)
    auc = roc_auc_score(y_te, te_prob)
    print(f"  Test  →  Acc: {acc:.3f} | F1: {f1:.3f} | AUC: {auc:.3f}")
    print(classification_report(y_te, y_pred, target_names=le.classes_, zero_division=0))
    fold_results.append((acc, f1, auc))

    # Track globally best model across all folds
    val_auc = roc_auc_score(y_va, va_prob)

    if val_auc > best_global_auc:
        best_global_auc = val_auc
        best_global_state = copy.deepcopy(net.state_dict())
        best_global_fold = fold + 1
        best_global_thr = best_thr

        print(f"New global best! Fold {best_global_fold} | Val AUC: {best_global_auc:.3f}")

    all_y_te.append(y_te)
    all_y_pred.append(y_pred)
    all_te_prob.append(te_prob)
    all_wt_te_embeddings.append(WT_te)
    all_mut_te_embeddings.append(MUT_te)

all_y_te = np.concatenate(all_y_te)
all_y_pred = np.concatenate(all_y_pred)
all_te_prob = np.concatenate(all_te_prob)
all_wt_te_embeddings = np.concatenate(all_wt_te_embeddings)
all_mut_te_embeddings = np.concatenate(all_mut_te_embeddings)


# Final cross-validation summary
r = np.array(fold_results)
print("\n" + "="*48)
print("  5-Fold Cross-Validation ─ Final Summary")
print("="*48)
for name, col in zip(["Accuracy", "F1-score", "AUC-ROC"], r.T):
    print(f"  {name}: {col.mean():.3f} \u00b1 {col.std():.3f}")
print(f"  Best epoch (mean): {np.mean(fold_epochs):.1f} \u00b1 {np.std(fold_epochs):.1f}")
pd.DataFrame(fold_results, columns=["Accuracy", "F1", "AUC"]).to_csv("/content/cv_results.csv", index=False)
print("Fold results saved to cv_results.csv")


# =========================================================================
# FINAL EVALUATION ON 10% UNSEEN TEST SET (FOR FAIR COMPARISON)
# =========================================================================

# Fit final scalers using only the 90% CV data
# To ensure strict fairness and avoid Data Leakage
sc_final_emb = StandardScaler()
sc_final_emb.fit(WT)

sc_final_llr = StandardScaler()
sc_final_llr.fit(delta_llr)

print("\n" + "="*48)
print("  FINAL EVALUATION ON 10% HOLD-OUT TEST SET")
print("="*48)

# Load the best global model from the Folds
net.load_state_dict(best_global_state)
net.eval()

# Scale the 10% test set
WT_te_final_sc  = sc_final_emb.transform(WT_test_final)
MUT_te_final_sc = sc_final_emb.transform(MUT_test_final)
DL_te_final_sc  = sc_final_llr.transform(delta_llr_test_final)

# Convert to Tensors
WT_t  = torch.tensor(WT_te_final_sc).to(device)
MUT_t = torch.tensor(MUT_te_final_sc).to(device)
DL_t  = torch.tensor(DL_te_final_sc).to(device)

with torch.no_grad():
    final_te_prob = torch.sigmoid(net(WT_t, MUT_t, DL_t)[0]).cpu().numpy().ravel()

# Predictions (using the optimal threshold from the best fold)
final_y_pred = (final_te_prob > best_global_thr).astype(int)

test_acc = accuracy_score(y_test_final, final_y_pred)
test_f1  = f1_score(y_test_final, final_y_pred, zero_division=0)
test_auc = roc_auc_score(y_test_final, final_te_prob)

print(f"  Accuracy : {test_acc:.4f}")
print(f"  F1-Score : {test_f1:.4f}")
print(f"  AUC-ROC  : {test_auc:.4f}")
print(f"  Threshold: {best_global_thr:.4f} (from Fold {best_global_fold})")
print("="*48)

In [ ]:

# PLOTS ONLY ON 10% UNSEEN HOLD-OUT TEST SET

from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.calibration import calibration_curve
from sklearn.decomposition import PCA
import umap.umap_ as umap
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

final_labels_text = le.inverse_transform(y_test_final.astype(int))

In [ ]:
#  Visualization — PCA and UMAP of Delta Embeddings
delta_test_raw = MUT_test_final - WT_test_final

colors = ["#e74c3c" if l == "Pathogenic" else "#27ae60" for l in final_labels_text]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pca = PCA(n_components=2)
X_pca = pca.fit_transform(delta_test_raw)

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.7, s=35)
axes[0].set_title("PCA — Δ Embedding on Unseen Test Set")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")

reducer = umap.UMAP(n_neighbors=10, min_dist=0.1, metric="cosine", random_state=42)
X_umap = reducer.fit_transform(delta_test_raw)

axes[1].scatter(X_umap[:, 0], X_umap[:, 1], c=colors, alpha=0.7, s=35)
axes[1].set_title("UMAP — Δ Embedding on Unseen Test Set")
axes[1].set_xlabel("UMAP1")
axes[1].set_ylabel("UMAP2")

from matplotlib.patches import Patch
axes[1].legend(handles=[
    Patch(color="#e74c3c", label="Pathogenic"),
    Patch(color="#27ae60", label="Benign"),
], loc="upper right")

plt.tight_layout()
plt.savefig("/content/embeddings_visualization_unseen_test.png", dpi=150)
plt.show()


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_final, final_y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix — Unseen Test Set")
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test_final, final_te_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Unseen Test Set")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Precision-Recall Curve

precision, recall, _ = precision_recall_curve(y_test_final, final_te_prob)
avg_precision = average_precision_score(y_test_final, final_te_prob)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, lw=2, label=f"AP = {avg_precision:.3f}")
plt.axhline(y_test_final.mean(), linestyle="--", label=f"Baseline = {y_test_final.mean():.2f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Unseen Test Set")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ΔLLR Distribution on unseen test only

df_delta_llr_test = pd.DataFrame({
    "ΔLLR": delta_llr_test_final.ravel(),
    "Label": final_labels_text
})

plt.figure(figsize=(10, 7))
sns.histplot(
    data=df_delta_llr_test,
    x="ΔLLR",
    hue="Label",
    kde=True,
    alpha=0.6
)
plt.title("ΔLLR Distribution — Unseen Test Set")
plt.xlabel("ΔLLR")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Calibration Plot

fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test_final,
    final_te_prob,
    n_bins=8,
    strategy="uniform"
)

plt.figure(figsize=(8, 6))
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Siamese Model")
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
plt.xlabel("Mean predicted probability")
plt.ylabel("Fraction of positives")
plt.title("Calibration Plot — Unseen Test Set")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# WT–MUT Euclidean Distance Distribution on unseen test only

distances_test = np.linalg.norm(WT_te_final_sc - MUT_te_final_sc, axis=1)

df_distances_test = pd.DataFrame({
    "Distance": distances_test,
    "Label": final_labels_text
})

plt.figure(figsize=(10, 7))
sns.histplot(
    data=df_distances_test,
    x="Distance",
    hue="Label",
    kde=True,
    alpha=0.6
)
plt.title("WT–MUT Embedding Distance Distribution — Unseen Test Set")
plt.xlabel("Euclidean Distance")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
#  Save Final Scalers for Inference
#
#  Two separate scalers are saved:
#  - sc_final_emb : fitted on WT embeddings only (consistent with CV)
#  - sc_final_llr : fitted on delta_llr values
#
#  During inference, both WT and MUT embeddings should be
#  transformed using sc_final_emb (same scaler for both).
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler

# Final embedding scaler — fit ONLY on WT (consistent with CV loop)
sc_final_emb = StandardScaler()
sc_final_emb.fit(WT)

with open("/content/scaler_final_emb.pkl", "wb") as f:
    pickle.dump(sc_final_emb, f)
print("Final Embedding Scaler saved.")

# Final LLR scaler — fit on delta_llr
sc_final_llr = StandardScaler()
sc_final_llr.fit(delta_llr)

with open("/content/scaler_final_llr.pkl", "wb") as f:
    pickle.dump(sc_final_llr, f)
print("Final Delta LLR Scaler saved.")

#Weights
torch.save({
    "model_state_dict" : best_global_state,
    "best_fold"        : best_global_fold,
    "best_auc"         : best_global_auc,
    "architecture"     : {"in_dim": 768, "hidden": 256},
}, "/content/siamese_model_weights.pth")

print(f"Best model saved — Fold {best_global_fold} | AUC: {best_global_auc:.3f}")

print("Trained model weights saved.")

# Verification
print(f"Embedding scaler mean shape : {sc_final_emb.mean_.shape}  ← expected (768,)")
print(f"LLR scaler mean                 : {sc_final_llr.mean_[0]:.4f}")

In [ ]:

#!zip -r /content.zip /content

#from google.colab import files
#files.download('/content.zip')